# Rescore Lorentzian fits with trace distance, all 14 states
Ported from `03B_rescore_tracedist_allstates.ipynb`. Reads the `H,C` / `H,J1..J4` already fitted by `01_E_Kossakowski_CONSTR_train-test_Lorentzian.ipynb` / `01_E_Lindblad_CONSTR_train-test_Lorentzian.ipynb` (does not refit), re-simulates each of the 14 states, and scores with trace distance.

**Before running:** fill in `kossak_fitted` / `lindblad_fitted` below with the actual output filenames produced by the two `01_E_*_Lorentzian` fitting notebooks.

In [ ]:
include("LiPoSID.jl")
using QuantumOptics
basis = NLevelBasis(2)
using LinearAlgebra
using HDF5
using Dates
using Statistics

In [ ]:
# Compatibility fix: newer DiffEqBase removed promote_dual
import DiffEqBase
if !isdefined(DiffEqBase, :promote_dual)
    @eval DiffEqBase promote_dual(T, ::Type{S}) where {S} = T
end

In [ ]:
σˣ = [ 0 1
       1 0 ]
σʸ = [ 0.   im*1
      -im*1  0   ]
σᶻ = [ 1.  0
       0  -1 ]

fᴷ₁ = σˣ/2;  fᴷ₂ = σʸ/2;  fᴷ₃ = σᶻ/2
fᴷᴼᴺᴮ = [fᴷ₁, fᴷ₂, fᴷ₃]

In [ ]:
widths    = ["wid1", "wid2", "wid3", "wid4", "wid5"]
all_states = vcat(["State0","State1","StateX","StateY"], ["Dodeca$i" for i in 1:10])   # 14 states
data_dir  = "DATA/"

# EDIT these to match the files produced by the 01_E_*_Lorentzian fitting notebooks:
kossak_fitted   = "E_KOSSAK_LORENTZIAN_CONSTR_TSSOS_treshold_1e-15_FROB_QO_2026-Sep-19_at_19-31.h5"
lindblad_fitted = "E_LINDBLAD4_LORENTZIAN_CONSTR_TSSOS_treshold_1e-9_FROB_QO_2026-Sep-19_at_19-31.h5"

date_str = Dates.format(today(), "yyyy-mm-dd")

In [ ]:
# ── Kossakowski rescore ────────────────────────────────────────────────────
out_kossak = "E_KOSSAK_LORENTZIAN_TRACEDIST_ALLSTATES_"*date_str*".h5"

h5open(out_kossak, "cw") do out
    h5open(kossak_fitted, "r") do fitted
        for widᵢ in widths
            println("\nKossak  width = ", widᵢ)
            H = convert.(ComplexF64, read(fitted[widᵢ]["H"]))
            C = convert.(ComplexF64, read(fitted[widᵢ]["C"]))
            Hˢⁱᵈ = DenseOperator(basis, H)
            eff_L = LiPoSID.get_lindblad_operators(C, fᴷᴼᴺᴮ)
            ops   = [DenseOperator(basis, j) for j in eff_L]
            wid_grp = create_group(out, widᵢ)
            for state in all_states
                tₛ, ρₛ = LiPoSID.read_lorentzian_timeevolution(data_dir, state, widᵢ)
                ρₛ   = convert(Vector{Matrix{ComplexF64}}, ρₛ)
                tᵗˢᵗ = Float64.(tₛ)
                ρₒ   = DenseOperator(basis, ρₛ[1])
                tout, ρ_t = timeevolution.master(tᵗˢᵗ, ρₒ, Hˢⁱᵈ, ops)
                ρˢⁱᵈ = [x.data for x in ρ_t]
                @assert length(ρₛ) == length(ρˢⁱᵈ) "length mismatch wid=$widᵢ state=$state"
                td = LiPoSID.TrDist_series(ρₛ, ρˢⁱᵈ)
                sg = create_group(wid_grp, state)
                sg["TraceDist"] = convert.(Float64, td)
                sg["time"]      = tᵗˢᵗ
                println("  ", state, "  max=", round(maximum(td), digits=5))
            end
        end
    end
end
println("\nSaved: ", out_kossak)

In [ ]:
# ── Lindblad rescore ───────────────────────────────────────────────────────
out_lindblad = "E_LINDBLAD_LORENTZIAN_TRACEDIST_ALLSTATES_"*date_str*".h5"

h5open(out_lindblad, "cw") do out
    h5open(lindblad_fitted, "r") do fitted
        for widᵢ in widths
            println("\nLindblad  width = ", widᵢ)
            H  = convert.(ComplexF64, read(fitted[widᵢ]["H"]))
            J1 = convert.(ComplexF64, read(fitted[widᵢ]["J1"]))
            J2 = convert.(ComplexF64, read(fitted[widᵢ]["J2"]))
            J3 = convert.(ComplexF64, read(fitted[widᵢ]["J3"]))
            J4 = convert.(ComplexF64, read(fitted[widᵢ]["J4"]))
            Hˢⁱᵈ = DenseOperator(basis, H)
            ops  = [DenseOperator(basis, j) for j in (J1, J2, J3, J4)]
            wid_grp = create_group(out, widᵢ)
            for state in all_states
                tₛ, ρₛ = LiPoSID.read_lorentzian_timeevolution(data_dir, state, widᵢ)
                ρₛ   = convert(Vector{Matrix{ComplexF64}}, ρₛ)
                tᵗˢᵗ = Float64.(tₛ)
                ρₒ   = DenseOperator(basis, ρₛ[1])
                tout, ρ_t = timeevolution.master(tᵗˢᵗ, ρₒ, Hˢⁱᵈ, ops)
                ρˢⁱᵈ = [x.data for x in ρ_t]
                @assert length(ρₛ) == length(ρˢⁱᵈ) "length mismatch wid=$widᵢ state=$state"
                td = LiPoSID.TrDist_series(ρₛ, ρˢⁱᵈ)
                sg = create_group(wid_grp, state)
                sg["TraceDist"] = convert.(Float64, td)
                sg["time"]      = tᵗˢᵗ
                println("  ", state, "  max=", round(maximum(td), digits=5))
            end
        end
    end
end
println("\nSaved: ", out_lindblad)